# LangChain Using Tools

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
model = "gpt-5.6-sol"

In [ ]:
llm = ChatOpenAI(
    model=model,
    temperature=0.0,
    max_tokens=500,
    timeout=500,
    reasoning_effort="low",
    max_retries=2,
)

In [ ]:
class ToolArgsInput(BaseModel):
    a: int = Field(..., description="The first number to add")
    b: int = Field(..., description="The second number to add")

In [ ]:
@tool("add", args_schema=ToolArgsInput, description="Add two numbers")
def add(a: int, b: int) -> int:
    """Add two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a + b

In [ ]:
@tool("multiply", args_schema=ToolArgsInput, description="Multiply two numbers")
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a * b

In [ ]:
tools = [add, multiply]
tool_map = {tool.name: tool for tool in tools}

In [ ]:
tool_map

In [ ]:
print(tool_map)

In [ ]:
class OutputSchema(BaseModel):
    """Schema for response."""

    answer: str
    justification: str

In [ ]:
llm_with_tools = llm.bind_tools(tools, response_format=OutputSchema, strict=True)

In [ ]:
question = "What is the sum of 5 and 10, and then multiply the result by 2?"

In [ ]:
messages = [
    SystemMessage(
        """
            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        """
    ),
    HumanMessage(question),
]

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
ai_message

In [ ]:
ai_message.tool_calls

In [ ]:
messages.append(ai_message)

In [ ]:
messages

In [ ]:
for message in messages:
    message.pretty_print()

In [ ]:
# parsed_tool_calls = parse_tool_calls(ai_message.additional_kwargs.get("tool_calls"))

In [ ]:
# parsed_tool_calls

In [ ]:
for tool_call in ai_message.tool_calls:
    tool_call_id = tool_call["id"]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [ ]:
messages

In [ ]:
for message in messages:
    message.pretty_print()

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
ai_message

In [ ]:
for tool_call in ai_message.tool_calls:
    tool_call_id = tool_call["id"]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [ ]:
for message in messages:
    message.pretty_print()

In [ ]:
messages